In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.attacks.fgsm import fgsm_attack
from src.attacks.pgd import pgd_attack

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\arvin\Desktop\projects\adversarial-ids


In [2]:
X_test = pd.read_csv(
    PROJECT_ROOT /
    "data" /
    "splits" /
    "X_test.csv"
)

y_test = pd.read_csv(
    PROJECT_ROOT /
    "data" /
    "splits" /
    "y_test.csv"
).squeeze()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nClass distribution:")
print(y_test.value_counts())

X_test shape: (378120, 70)
y_test shape: (378120,)

Class distribution:
target
0    314259
1     63861
Name: count, dtype: int64


In [3]:
X_test_tensor = torch.tensor(
    X_test.values,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1, 1)

print("X tensor:", X_test_tensor.shape)
print("y tensor:", y_test_tensor.shape)

X tensor: torch.Size([378120, 70])
y tensor: torch.Size([378120, 1])


In [4]:
class IDSNeuralNetwork(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [5]:
input_size = X_test.shape[1]

baseline_model = IDSNeuralNetwork(input_size)

baseline_model.load_state_dict(
    torch.load(
        PROJECT_ROOT /
        "models" /
        "neural_network_baseline.pth",
        map_location="cpu"
    )
)

baseline_model.eval()


adv_model = IDSNeuralNetwork(input_size)

adv_model.load_state_dict(
    torch.load(
        PROJECT_ROOT /
        "models" /
        "neural_network_adversarial_fgsm.pth",
        map_location="cpu"
    )
)

adv_model.eval()

print("Baseline NN loaded.")
print("Adversarially trained NN loaded.")

Baseline NN loaded.
Adversarially trained NN loaded.


In [6]:
loss_fn = nn.BCEWithLogitsLoss()

In [7]:
def calculate_metrics(y_true, y_prob, threshold=0.5):

    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0
    )

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "FPR": fpr,
        "FNR": fnr,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

In [8]:
def calculate_attack_success_rate(
    y_true,
    clean_pred,
    adv_pred
):

    y_true = np.asarray(y_true).astype(int)
    clean_pred = np.asarray(clean_pred).astype(int)
    adv_pred = np.asarray(adv_pred).astype(int)

    correctly_detected_attacks = (
        (y_true == 1) &
        (clean_pred == 1)
    )

    total = correctly_detected_attacks.sum()

    if total == 0:
        return 0.0

    successful_evasions = (
        correctly_detected_attacks &
        (adv_pred == 0)
    ).sum()

    return successful_evasions / total

In [9]:
with torch.no_grad():

    baseline_logits = baseline_model(
        X_test_tensor
    )

    baseline_prob = torch.sigmoid(
        baseline_logits
    ).numpy().ravel()

baseline_clean_metrics = calculate_metrics(
    y_test.values,
    baseline_prob
)

baseline_clean_pred = (
    baseline_prob >= 0.5
).astype(int)

print("BASELINE NN — CLEAN TEST")
print(baseline_clean_metrics)

BASELINE NN — CLEAN TEST
{'Accuracy': 0.9838463979688987, 'Precision': 0.9612933113947507, 'Recall': 0.9422965503202267, 'F1': 0.951700142337498, 'ROC-AUC': 0.9986783747524243, 'PR-AUC': 0.993938458142201, 'FPR': np.float64(0.007710200821615292), 'FNR': np.float64(0.057703449679773255), 'TN': np.int64(311836), 'FP': np.int64(2423), 'FN': np.int64(3685), 'TP': np.int64(60176)}


In [10]:
with torch.no_grad():

    adv_logits = adv_model(
        X_test_tensor
    )

    adv_prob = torch.sigmoid(
        adv_logits
    ).numpy().ravel()

adv_clean_metrics = calculate_metrics(
    y_test.values,
    adv_prob
)

adv_clean_pred = (
    adv_prob >= 0.5
).astype(int)

print("ADVERSARIALLY TRAINED NN — CLEAN TEST")
print(adv_clean_metrics)

ADVERSARIALLY TRAINED NN — CLEAN TEST
{'Accuracy': 0.9816037236856024, 'Precision': 0.9722641791292512, 'Recall': 0.9172421352625233, 'F1': 0.9439520417700712, 'ROC-AUC': 0.9984189151877736, 'PR-AUC': 0.9928753211914687, 'FPR': np.float64(0.005317270149780913), 'FNR': np.float64(0.0827578647374767), 'TN': np.int64(312588), 'FP': np.int64(1671), 'FN': np.int64(5285), 'TP': np.int64(58576)}


In [11]:
X_test_fgsm_baseline = fgsm_attack(
    model=baseline_model,
    x=X_test_tensor,
    y=y_test_tensor,
    epsilon=0.01,
    loss_fn=loss_fn
)

In [12]:
with torch.no_grad():

    logits = baseline_model(
        X_test_fgsm_baseline
    )

    prob = torch.sigmoid(
        logits
    ).numpy().ravel()

baseline_fgsm_metrics = calculate_metrics(
    y_test.values,
    prob
)

baseline_fgsm_pred = (
    prob >= 0.5
).astype(int
)

baseline_fgsm_asr = calculate_attack_success_rate(
    y_test.values,
    baseline_clean_pred,
    baseline_fgsm_pred
)

print("BASELINE NN — FGSM ε=0.01")
print(baseline_fgsm_metrics)
print("Attack Success Rate:", baseline_fgsm_asr)

BASELINE NN — FGSM ε=0.01
{'Accuracy': 0.8942584364751931, 'Precision': 0.6795284352350305, 'Recall': 0.7076306352860118, 'F1': 0.6932948766137631, 'ROC-AUC': 0.9527105798631808, 'PR-AUC': 0.8275338035167222, 'FPR': np.float64(0.06781667350815729), 'FNR': np.float64(0.29236936471398817), 'TN': np.int64(292947), 'FP': np.int64(21312), 'FN': np.int64(18671), 'TP': np.int64(45190)}
Attack Success Rate: 0.24903616059558628


In [13]:
X_test_pgd_baseline = pgd_attack(
    model=baseline_model,
    x=X_test_tensor,
    y=y_test_tensor,
    epsilon=0.01,
    alpha=0.001,
    num_steps=10,
    loss_fn=loss_fn
)

In [14]:
with torch.no_grad():

    logits = baseline_model(
        X_test_pgd_baseline
    )

    prob = torch.sigmoid(
        logits
    ).numpy().ravel()

baseline_pgd_metrics = calculate_metrics(
    y_test.values,
    prob
)

baseline_pgd_pred = (
    prob >= 0.5
).astype(int
)

baseline_pgd_asr = calculate_attack_success_rate(
    y_test.values,
    baseline_clean_pred,
    baseline_pgd_pred
)

print("BASELINE NN — PGD ε=0.01")
print(baseline_pgd_metrics)
print("Attack Success Rate:", baseline_pgd_asr)

BASELINE NN — PGD ε=0.01
{'Accuracy': 0.8930736274198667, 'Precision': 0.6752221125370188, 'Recall': 0.7069103208531028, 'F1': 0.6907029582539647, 'ROC-AUC': 0.9518809507864201, 'PR-AUC': 0.8252591170551805, 'FPR': np.float64(0.06909587314921768), 'FNR': np.float64(0.29308967914689715), 'TN': np.int64(292545), 'FP': np.int64(21714), 'FN': np.int64(18717), 'TP': np.int64(45144)}
Attack Success Rate: 0.24980058495081095


In [15]:
X_test_fgsm_adv = fgsm_attack(
    model=adv_model,
    x=X_test_tensor,
    y=y_test_tensor,
    epsilon=0.01,
    loss_fn=loss_fn
)

In [16]:
with torch.no_grad():

    logits = adv_model(
        X_test_fgsm_adv
    )

    prob = torch.sigmoid(
        logits
    ).numpy().ravel()

adv_fgsm_metrics = calculate_metrics(
    y_test.values,
    prob
)

adv_fgsm_pred = (
    prob >= 0.5
).astype(int
)

adv_fgsm_asr = calculate_attack_success_rate(
    y_test.values,
    adv_clean_pred,
    adv_fgsm_pred
)

print("ADVERSARIALLY TRAINED NN — FGSM ε=0.01")
print(adv_fgsm_metrics)
print("Attack Success Rate:", adv_fgsm_asr)

ADVERSARIALLY TRAINED NN — FGSM ε=0.01
{'Accuracy': 0.9777610282450016, 'Precision': 0.9573888943878056, 'Recall': 0.9087706111711373, 'F1': 0.9324464367483672, 'ROC-AUC': 0.9973536593245922, 'PR-AUC': 0.9884834869156968, 'FPR': np.float64(0.008219335007111968), 'FNR': np.float64(0.09122938882886268), 'TN': np.int64(311676), 'FP': np.int64(2583), 'FN': np.int64(5826), 'TP': np.int64(58035)}
Attack Success Rate: 0.009235864517891287


In [17]:
X_test_pgd_adv = pgd_attack(
    model=adv_model,
    x=X_test_tensor,
    y=y_test_tensor,
    epsilon=0.01,
    alpha=0.001,
    num_steps=10,
    loss_fn=loss_fn
)

In [18]:
with torch.no_grad():

    logits = adv_model(
        X_test_pgd_adv
    )

    prob = torch.sigmoid(
        logits
    ).numpy().ravel()

adv_pgd_metrics = calculate_metrics(
    y_test.values,
    prob
)

adv_pgd_pred = (
    prob >= 0.5
).astype(int
)

adv_pgd_asr = calculate_attack_success_rate(
    y_test.values,
    adv_clean_pred,
    adv_pgd_pred
)

print("ADVERSARIALLY TRAINED NN — PGD ε=0.01")
print(adv_pgd_metrics)
print("Attack Success Rate:", adv_pgd_asr)

ADVERSARIALLY TRAINED NN — PGD ε=0.01
{'Accuracy': 0.9776843330159738, 'Precision': 0.9569763031612275, 'Recall': 0.9087236341429041, 'F1': 0.9322259883375368, 'ROC-AUC': 0.997160015893111, 'PR-AUC': 0.9876857924782435, 'FPR': np.float64(0.008302069312255179), 'FNR': np.float64(0.09127636585709588), 'TN': np.int64(311650), 'FP': np.int64(2609), 'FN': np.int64(5829), 'TP': np.int64(58032)}
Attack Success Rate: 0.00928708003277793


In [19]:
final_results = pd.DataFrame([
    {
        "Model": "Baseline NN",
        "Condition": "Clean",
        **baseline_clean_metrics,
        "Attack Success Rate": np.nan
    },
    {
        "Model": "Baseline NN",
        "Condition": "FGSM",
        **baseline_fgsm_metrics,
        "Attack Success Rate": baseline_fgsm_asr
    },
    {
        "Model": "Baseline NN",
        "Condition": "PGD",
        **baseline_pgd_metrics,
        "Attack Success Rate": baseline_pgd_asr
    },
    {
        "Model": "Adversarially Trained NN",
        "Condition": "Clean",
        **adv_clean_metrics,
        "Attack Success Rate": np.nan
    },
    {
        "Model": "Adversarially Trained NN",
        "Condition": "FGSM",
        **adv_fgsm_metrics,
        "Attack Success Rate": adv_fgsm_asr
    },
    {
        "Model": "Adversarially Trained NN",
        "Condition": "PGD",
        **adv_pgd_metrics,
        "Attack Success Rate": adv_pgd_asr
    }
])

final_results

,Model,Condition,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,FPR,FNR,TN,FP,FN,TP,Attack Success Rate
0,Baseline NN,Clean,0.983846,0.961293,0.942297,0.951700,0.998678,0.993938,0.007710,0.057703,311836,2423,3685,60176,NaN
1,Baseline NN,FGSM,0.894258,0.679528,0.707631,0.693295,0.952711,0.827534,0.067817,0.292369,292947,21312,18671,45190,0.249036
2,Baseline NN,PGD,0.893074,0.675222,0.706910,0.690703,0.951881,0.825259,0.069096,0.293090,292545,21714,18717,45144,0.249801
3,Adversarially Trained NN,Clean,0.981604,0.972264,0.917242,0.943952,0.998419,0.992875,0.005317,0.082758,312588,1671,5285,58576,NaN
4,Adversarially Trained NN,FGSM,0.977761,0.957389,0.908771,0.932446,0.997354,0.988483,0.008219,0.091229,311676,2583,5826,58035,0.009236
5,Adversarially Trained NN,PGD,0.977684,0.956976,0.908724,0.932226,0.997160,0.987686,0.008302,0.091276,311650,2609,5829,58032,0.009287


In [20]:
results_dir = (
    PROJECT_ROOT /
    "experiments" /
    "results"
)

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

final_results.to_csv(
    results_dir /
    "final_test_results.csv",
    index=False
)

print(
    "Saved:",
    results_dir / "final_test_results.csv"
)

Saved: c:\Users\arvin\Desktop\projects\adversarial-ids\experiments\results\final_test_results.csv


In [21]:
fgsm_asr_reduction = (
    baseline_fgsm_asr -
    adv_fgsm_asr
)

pgd_asr_reduction = (
    baseline_pgd_asr -
    adv_pgd_asr
)

print(
    "FGSM ASR reduction:",
    fgsm_asr_reduction
)

print(
    "PGD ASR reduction:",
    pgd_asr_reduction
)

FGSM ASR reduction: 0.23980029607769499
PGD ASR reduction: 0.24051350491803303


In [22]:
fgsm_asr_reduction_pct = (
    (baseline_fgsm_asr - adv_fgsm_asr)
    / baseline_fgsm_asr
) * 100

pgd_asr_reduction_pct = (
    (baseline_pgd_asr - adv_pgd_asr)
    / baseline_pgd_asr
) * 100

print(
    "FGSM ASR reduction (%):",
    fgsm_asr_reduction_pct
)

print(
    "PGD ASR reduction (%):",
    pgd_asr_reduction_pct
)

FGSM ASR reduction (%): 96.291356044117
PGD ASR reduction (%): 96.2822024477618


In [23]:
EPSILONS = [
    0.001,
    0.005,
    0.01,
    0.05,
    0.1
]